In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import os
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import argparse
from pathlib import Path

In [ ]:
# Plots


def load_pi_r(emp_pi_r):
    pi_r = emp_pi_r
    pi_r = np.array([1-sum(pi_r)] + list(pi_r))
    df = filter_N_upstream_df[["ze2010","pi_r"]].drop_duplicates()
    df.loc[~df.pi_r.isna(),"sim_pi_r"] = pi_r
    df.pi_r.fillna(0,inplace = True)
    df.sim_pi_r.fillna(0,inplace = True)
    return df



def plot_downstream(df,col_name,ax):

    tmp = france.merge(df,on = "ze2010",how = "left")
    tmp[col_name].fillna(0,inplace = True)
    if col_name != "productivity":
        vmin = min(df.query('pi_r >0')["pi_r"].to_list()+df.query('sim_pi_r >0')["sim_pi_r"].to_list())
        vmax = max(df["pi_r"].to_list()+df["sim_pi_r"].to_list())
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    else: 
        vmin,vmax = min(df.query('productivity >0').productivity),max(df['productivity'])
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        print(df.productivity.describe())

    tmp.query(col_name+' == 0').plot(color = "gray",ax=ax)
    tmp.query(col_name+' > 0').plot(column = col_name,ax=ax,norm = norm, cmap = "viridis",edgecolor ="black")

    ax_inset = inset_axes(ax, width="65%", height="65%", loc="upper right",bbox_to_anchor=(0.745, 0.775, 0.25, 0.25), bbox_transform=ax.transAxes, borderpad=1)  
    data_idf = tmp[tmp['ze2010'].isin(idf_ze)]
    data_idf.query(col_name+' == 0').plot(color = "gray",ax=ax_inset)
    data_idf.query(col_name+' > 0').plot(column = col_name,ax=ax_inset,norm = norm, cmap = "viridis",edgecolor ="black")

    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax.set_xlim(-5, 10)
    ax.set_ylim(42, 52)
    ax.set_xticks([])
    ax.set_yticks([])

    add_colorbar(ax,norm)
    return norm

def load_productivity(file_name):
    prod = np.load(os.path.join(folder,f"{file_name}.npy"))
    df = filter_N_upstream_df[["ze2010","pi_r"]].drop_duplicates()
    df.loc[~df.pi_r.isna(),"productivity"] = prod
    df.productivity.fillna(0,inplace = True)
    return df

def add_colorbar(ax,norm):
    divider = make_axes_locatable(ax)
    cax = inset_axes(ax, width="80%", height="5%", loc='lower center', borderpad=-1.5)
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
    sm._A = []
    fig.colorbar(sm, cax=cax, orientation='horizontal')

def load_productivity(productivity):
    prod = productivity
    df = filter_N_upstream_df[["ze2010","pi_r"]].drop_duplicates()
    df.loc[~df.pi_r.isna(),"productivity"] = prod
    df.productivity.fillna(0,inplace = True)
    return df

In [ ]:
def unpack_simulated_moments(sim_moments, empirical_moments):
    keys = [
        "agg_labor_share",
        "agg_industry_share",
        "emp_gamma_ls",
        "reg_coef",
        "emp_pi_r"
    ]

    sizes = [m.size for m in empirical_moments]
    splits = np.cumsum(sizes)[:-1]
    blocks = np.split(sim_moments, splits, axis=0)

    return {
        k: b.reshape(m.shape + (b.shape[1],))
        for k, m, b in zip(keys, empirical_moments, blocks)
    }


def unpack_params(params, S = 9, R_downstream = 35):
    """
    This function takes a vector of parameters and returns the parameters
    as separated variables.
    """

    beta = params[0:5]
    labor_share_tech = params[5]

    input_share_tech = params[6:6+S]
    input_share_tech = input_share_tech / np.sum(input_share_tech)

    productivity_ = params[6+S : 6+S+R_downstream]
    T_ = params[6+S+R_downstream :]

    return {"beta":beta, "labor_share_tech":labor_share_tech, "input_share_tech":input_share_tech, "productivity":productivity_, "T":T_}


In [ ]:
# Loading
# --- Paths ---
industry = "aero"  # or "auto_24"
input_folder = Path(f"../baseline_{industry}")

# --- Load only what is used ---
coefs = pd.read_csv(input_folder / "stats.csv")

agg_labor_share = coefs.loc[1, "value"]          # coefs[2,"value"] in Julia
epsilon = coefs.loc[0, "value"]                  # coefs[1,"value"]

agg_industry_share = np.load(input_folder / "input_share.npy")
emp_gamma_ls = np.load(input_folder / "emp_gamma_ls.npy").T
emp_pi_r = np.load(input_folder / "emp_pi_r.npy")[1:]
reg_coef = np.load(input_folder / "reg_coef.npy")

# --- Empirical moments (same ordering as Julia) ---
reference_empirical_moments = [
    np.array([agg_labor_share]),
    agg_industry_share[1:],   # Julia [2:end]
    emp_gamma_ls,
    reg_coef,
    emp_pi_r
]

In [ ]:
# Importation 

folder = "../reporting_aero/"
empirical_moments = np.load(os.path.join(folder,f"empirical_moments.npy"))
best_simulated_moments = np.load(os.path.join(folder,f"best_simulated_moments.npy"))
best_params = np.load(os.path.join(folder,f"best_parameters_list.npy"))


best_simulated_moments_dict = unpack_simulated_moments(best_simulated_moments, reference_empirical_moments)
empirical_moments_dict = unpack_simulated_moments(empirical_moments.T, reference_empirical_moments)
K = 1

In [ ]:
pd.DataFrame(unpack_params(best_params)['T']).describe()

# Loss function

In [ ]:
loss_function = np.sum((empirical_moments.T-best_simulated_moments)**2,axis = 0)
loss_function/=loss_function[0]/100


plt.plot(loss_function)
plt.title('Loss function')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2, axis=0))

moment_names = list(empirical_moments_dict.keys())
loss_dict = {}

# --- Compute RMSE and normalize ---
for key in moment_names:
    emp = empirical_moments_dict[key]
    sim = best_simulated_moments_dict[key]  # shape: (*shape_emp, n_sim)
    
    emp_flat = emp.ravel()[:, np.newaxis]          # shape (K, 1)
    sim_flat = sim.reshape(-1, sim.shape[-1])     # shape (K, n_sim)
    
    loss = rmse(emp_flat, sim_flat)
    loss_norm = 100 * loss / loss[0]  # normalize to 100% at stage 0
    loss_dict[key] = loss_norm

# --- Plot in 3x2 subplots ---
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
axes = axes.flatten()
n_stages = len(next(iter(loss_dict.values())))  # number of stages

for i, key in enumerate(moment_names):
    ax = axes[i]
    ax.plot(loss_dict[key], marker='o')
    ax.set_title(key)
    ax.set_xlabel("Simulation / Stage")
    ax.set_ylabel("RMSE (%)")
    ax.grid(False)
    
    # Vertical dashed lines every 3 stages, starting at stage 1
    for vline in range(1, n_stages, 3):
        ax.axvline(vline, color='gray', linestyle='--', alpha=0.6)

# Hide unused subplot if less than 6 moments
if len(moment_names) < 6:
    axes[-1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def squared_error(y_true, y_pred):
    """Compute sum of squared errors along all spatial dimensions."""
    return np.sum((y_true.ravel()[:, np.newaxis] - y_pred.reshape(-1, y_pred.shape[-1]))**2, axis=0)

moment_names = list(empirical_moments_dict.keys())
n_stages = next(iter(best_simulated_moments_dict.values())).shape[-1]

# --- Compute contribution of each type ---
contrib_dict = {key: np.zeros(n_stages) for key in moment_names}
total_se = np.zeros(n_stages)

for key in moment_names:
    emp = empirical_moments_dict[key]
    sim = best_simulated_moments_dict[key]  # shape: (*shape_emp, n_stages)
    
    se = squared_error(emp, sim)  # shape: (n_stages,)
    contrib_dict[key] = se
    total_se += se

# --- Compute % contribution ---
for key in moment_names:
    contrib_dict[key] = 100 * contrib_dict[key] / total_se

# --- Plot ---
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
axes = axes.flatten()

for i, key in enumerate(moment_names):
    ax = axes[i]
    ax.plot(contrib_dict[key], marker='o')
    ax.set_title(f"{key} contribution to total RMSE (%)")
    ax.set_xlabel("Simulation / Stage")
    ax.set_ylabel("Contribution (%)")
    ax.set_ylim(0, 100)
    ax.grid(True)
    
    # Vertical dashed lines every 3 stages
    for vline in range(1, n_stages, 3):
        ax.axvline(vline, color='gray', linestyle='--', alpha=0.5)

# Hide unused subplot if less than 6 moments
if len(moment_names) < 6:
    axes[-1].axis('off')

plt.tight_layout()
plt.show()


# Gamma_ls 

In [ ]:
K = -1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

emp = empirical_moments_dict['emp_gamma_ls']
best = best_simulated_moments_dict['emp_gamma_ls'][:, :, K]

x = emp.ravel()
y = best.ravel()

# Keep only non-zero empirical observations
mask = (x != 0)
x, y = x[mask], y[mask]

# Bubble sizes (scaled for visibility)
sizes = 300 * (x / x.max())

# Threshold for top 20% empirical values
threshold = np.percentile(x, 80)
top_mask = x >= threshold

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---- Subplot 1: Full bubble plot ----
axes[0].scatter(
    x, y,
    s=sizes,
    alpha=0.5,
    edgecolor="black"
)

lims_full = [
    min(x.min(), y.min()),
    max(x.max(), y.max())
]
axes[0].plot(lims_full, lims_full, color="black")
axes[0].set_xlim(lims_full)
axes[0].set_ylim(lims_full)

axes[0].set_xlabel(r"Empirical $\gamma_{ls}$")
axes[0].set_ylabel(r"Simulated $\gamma_{ls}$")
axes[0].set_title("Bubble plot – all observations")

# ---- Subplot 2: Top 20% empirical bubble plot ----
axes[1].scatter(
    x[top_mask], y[top_mask],
    s=sizes[top_mask],
    alpha=0.5,
    edgecolor="black"
)

lims_top = [
    min(x[top_mask].min(), y[top_mask].min()),
    max(x[top_mask].max(), y[top_mask].max())
]
axes[1].plot(lims_top, lims_top, color="black")
axes[1].set_xlim(lims_top)
axes[1].set_ylim(lims_top)

axes[1].set_xlabel(r"Empirical $\gamma_{ls}$")
axes[1].set_ylabel(r"Simulated $\gamma_{ls}$")
axes[1].set_title("Bubble plot – top 20% empirical")

plt.tight_layout()
plt.show()


In [ ]:
empirical_moments_dict

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.distributions.empirical_distribution import ECDF

# --- Data ---
emp = empirical_moments_dict['emp_gamma_ls'].ravel()
sim = best_simulated_moments_dict['emp_gamma_ls'][:, :, K].ravel()

# keep positive (and non-zero) moments
emp_nz = emp[emp >= 0.01]
sim_nz = sim[sim >= 0.01]

# --- CDF (log-log) ---
xmin = min(emp_nz.min(), sim_nz.min())
xmax = max(emp_nz.max(), sim_nz.max())
x_vals = np.linspace(xmin, xmax, 300)
x_vals = x_vals[x_vals > 0]

F_emp = ECDF(emp_nz)
F_sim = ECDF(sim_nz)

cdf_emp = F_emp(x_vals)
cdf_sim = F_sim(x_vals)

keep = (cdf_emp > 0) & (cdf_sim > 0)

# --- Histogram threshold ---
x_chi = np.quantile(emp[emp != 0], 0.9)

emp_tail = emp[emp > x_chi]
sim_tail = sim[sim > x_chi]

bins = np.linspace(x_chi, max(emp_tail.max(), sim_tail.max()), 31)

# --- Plot ---
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# CDF
ax[0].plot(x_vals[keep], cdf_emp[keep], label="Empirical")
ax[0].plot(x_vals[keep], cdf_sim[keep], label="Simulated")
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_xlabel(r"$\gamma_{ls}$")
ax[0].set_ylabel("CDF")
ax[0].set_title("Log-Log CDF of $\gamma_{ls}$")
ax[0].legend()

# Distribution
ax[1].hist(emp_tail, bins=bins, alpha=0.5, label="Empirical")
ax[1].hist(sim_tail, bins=bins, alpha=0.5, label="Simulated")
ax[1].set_title(r"$\gamma_{ls}$ (upper tail)")
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
empirical_moments_dict

In [ ]:

# --- Take the positive, non-zero values ---
emp = empirical_moments_dict['emp_gamma_ls'].ravel()
sim = best_simulated_moments_dict['emp_gamma_ls'][:, :, -1].ravel()

mask = emp > 0.0  # keep regions with non-zero empirical gamma
emp_nz = emp[mask]
sim_nz = sim[mask]

# --- Create DataFrame ---
df = pd.DataFrame({
    "empirical_gamma_ls": emp_nz,
    "simulated_gamma_ls": sim_nz
})

# --- Describe distributions ---
description = df.describe()
print(description)


In [ ]:
tmp = pd.DataFrame(np.ndarray.flatten(empirical_moments_dict['emp_gamma_ls']),columns= ["emp_gamma_ls"])
print(tmp.query("emp_gamma_ls==0").shape)
tmp.query('emp_gamma_ls!= 0').describe()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import powerlaw

# ------------------------------------------------------------------
# 1. Extract non-zero, positive data
# ------------------------------------------------------------------
x = tmp.loc[tmp['emp_gamma_ls'] != 0, 'emp_gamma_ls'].values
x = x[x > 0]

# ------------------------------------------------------------------
# 2. Log–log CCDF (diagnostic)
# ------------------------------------------------------------------
x_sorted = np.sort(x)
ccdf = 1.0 - np.arange(1, len(x_sorted) + 1) / len(x_sorted)

plt.figure()
plt.loglog(x_sorted, ccdf, marker='.', linestyle='none')
plt.xlabel("emp_gamma_ls")
plt.ylabel("P(X ≥ x)")
plt.title("Log–log CCDF of non-zero emp_gamma_ls")
plt.show()

# ------------------------------------------------------------------
# 3. Power-law fit (MLE)
# ------------------------------------------------------------------
fit = powerlaw.Fit(x, discrete=False, verbose=False)

alpha = fit.power_law.alpha
xmin = fit.power_law.xmin

print(f"Power-law alpha: {alpha:.4f}")
print(f"Power-law xmin : {xmin:.4f}")

# ------------------------------------------------------------------
# 4. Goodness-of-fit: power law vs lognormal
# ------------------------------------------------------------------
R, p = fit.distribution_compare('power_law', 'lognormal')

print(f"Likelihood ratio (power law vs lognormal): R = {R:.4f}")
print(f"p-value: {p:.4f}")

# ------------------------------------------------------------------
# 5. Visual comparison of CCDFs
# ------------------------------------------------------------------
fig = fit.plot_ccdf(label='Empirical')
fit.power_law.plot_ccdf(ax=fig, linestyle='--', label='Power law')
fit.lognormal.plot_ccdf(ax=fig, linestyle=':', label='Lognormal')
plt.legend()
plt.show()


# Pi_r

In [ ]:
K = -1

industry = "aero"
folder = f"../baseline_{industry}/"

idf_ze = ['1101', '1111', '1102', '1104', '1118', '1115', '1116', '1105',
       '1117', '1110', '1119', '1112', '1103', '1109', '1106', '1114',
       '1113', '1108', '1107']

       
filter_N_upstream_df = pd.read_csv(os.path.join(folder,"filter_N_upstream.csv"))
filter_N_upstream_df.ze2010 = filter_N_upstream_df.ze2010.astype(str).str.zfill(4)


distances = np.load(os.path.join(folder,"full_distances.npy"))

france = gpd.read_file(os.path.join(folder,"france.shp")).sort_values(by = 'ze2010')

ref = pd.DataFrame(distances[:297,:297], index=france["ze2010"].values, columns=france["ze2010"].values)
ref.reset_index(inplace=True)
ref.rename(columns={'index': 'ze2010_i'}, inplace=True)
ref = ref.melt(id_vars='ze2010_i', var_name='ze2010_j', value_name='M_ij')




fig,axs = plt.subplots(2,2,figsize = (15,10))
pi_r = load_pi_r(best_simulated_moments_dict['emp_pi_r'][:,K])
prod = load_productivity(unpack_params(best_params[:,K])['productivity'])
plot_downstream(pi_r,"pi_r",axs[0,0])
plot_downstream(pi_r,"sim_pi_r",axs[0,1])
plot_downstream(prod,"productivity",axs[1,0])


axs[0,0].set_title(r'Empirical $\pi_{jA}$')
axs[0,1].set_title(r'Simulated $\pi_{jA}$')
axs[1,0].set_title(r'Simulated $A_i$')

x = pi_r.pi_r.values
y = pi_r.sim_pi_r.values

mask = x > 0
x, y = x[mask], y[mask]

# Bubble sizes proportional to empirical pi_r
sizes = 400 * (x / x.max())

axs[1, 1].scatter(x, y, s=sizes, alpha=0.6)

lims = [
    min(x.min(), y.min()),
    max(x.max(), y.max())
]
axs[1, 1].plot(lims, lims, color="black")
axs[1, 1].set_xlim(lims)
axs[1, 1].set_ylim(lims)

axs[1, 1].set_xlabel(r"Empirical $\pi_{jA}$")
axs[1, 1].set_ylabel(r"Simulated $\pi_{jA}$")
axs[1, 1].set_title(r"Empirical vs Simulated $\pi_{jA}$")

plt.tight_layout()
plt.show()


In [ ]:
pd.DataFrame(unpack_params(best_params)['productivity']).describe()

In [ ]:
pd.DataFrame(unpack_params(best_params)['productivity']).describe()